In [0]:
'''
Passo: configurar ambiente e paths para publicação progressiva de lotes.
O que observar: reutiliza os mesmos volumes criados pelo notebook gerador.
Validar: os paths de staging e inbox devem apontar para os volumes corretos.
Sinal de erro: apontar para paths inexistentes ou inverter staging com inbox.
'''

from pyspark.sql import functions as F

CATALOG = 'aulas_ao_vivo'
SCHEMA = 'live_20260413'
VOLUME_STAGING = 'armazenamento_pedidos_staging'
VOLUME_INBOX = 'armazenamento_pedidos_inbox'

STAGING_PATH = f'/Volumes/{CATALOG}/{SCHEMA}/{VOLUME_STAGING}'
INBOX_PATH = f'/Volumes/{CATALOG}/{SCHEMA}/{VOLUME_INBOX}'

JSON_STAGING_PATH = f'{STAGING_PATH}/pedidos/json'
PARQUET_STAGING_PATH = f'{STAGING_PATH}/pedidos/parquet'
JSON_INBOX_PATH = f'{INBOX_PATH}/pedidos/json'
PARQUET_INBOX_PATH = f'{INBOX_PATH}/pedidos/parquet'

print(f'Staging JSON:   {JSON_STAGING_PATH}')
print(f'Staging Parquet:{PARQUET_STAGING_PATH}')
print(f'Inbox JSON:     {JSON_INBOX_PATH}')
print(f'Inbox Parquet:  {PARQUET_INBOX_PATH}')

In [0]:
'''
Passo: criar função para detectar o último lote já publicado no inbox.
O que observar: a função extrai o número do lote das pastas existentes no inbox.
Validar: retorna 0 se o inbox estiver vazio, ou o maior número de lote encontrado.
Sinal de erro: não reconhecer o padrão "lote_XXX" e falhar na detecção.
'''

def detectar_ultimo_lote_publicado(formato: str) -> int:
    '''Detecta o último lote publicado no inbox para um formato específico.
    
    Args:
        formato: 'json' ou 'parquet'
    
    Returns:
        Número do último lote publicado, ou 0 se inbox estiver vazio.
    '''
    if formato not in ['json', 'parquet']:
        raise ValueError("Formato deve ser 'json' ou 'parquet'")
    
    inbox_path = JSON_INBOX_PATH if formato == 'json' else PARQUET_INBOX_PATH
    
    try:
        lotes = [
            item.name.rstrip('/')
            for item in dbutils.fs.ls(inbox_path)
            if item.name.startswith('lote_')
        ]
        
        if not lotes:
            return 0
        
        # Extrai números dos lotes (ex: "lote_001" -> 1)
        numeros = [int(lote.split('_')[1]) for lote in lotes]
        return max(numeros)
    
    except Exception:
        # Inbox vazio ou não existe
        return 0

# Teste
print(f"Último lote JSON publicado: {detectar_ultimo_lote_publicado('json')}")
print(f"Último lote Parquet publicado: {detectar_ultimo_lote_publicado('parquet')}")

In [0]:
'''
Passo: criar função para listar lotes disponíveis no staging que ainda não foram publicados.
O que observar: compara os lotes no staging com os já publicados no inbox.
Validar: retorna lista ordenada dos lotes pendentes de publicação.
Sinal de erro: não respeitar a ordem dos lotes e publicar fora de sequência.
'''

def detectar_lotes_disponiveis(formato: str) -> list:
    '''Detecta lotes disponíveis no staging que ainda não foram publicados no inbox.
    
    Args:
        formato: 'json' ou 'parquet'
    
    Returns:
        Lista ordenada de números de lotes disponíveis para publicação.
    '''
    if formato not in ['json', 'parquet']:
        raise ValueError("Formato deve ser 'json' ou 'parquet'")
    
    staging_path = JSON_STAGING_PATH if formato == 'json' else PARQUET_STAGING_PATH
    
    try:
        # Lista todos os lotes no staging
        lotes_staging = [
            item.name.rstrip('/')
            for item in dbutils.fs.ls(staging_path)
            if item.name.startswith('lote_')
        ]
        
        if not lotes_staging:
            return []
        
        # Extrai números dos lotes
        numeros_staging = sorted([int(lote.split('_')[1]) for lote in lotes_staging])
        
        # Detecta o último lote já publicado
        ultimo_publicado = detectar_ultimo_lote_publicado(formato)
        
        # Retorna apenas lotes maiores que o último publicado
        return [num for num in numeros_staging if num > ultimo_publicado]
    
    except Exception as e:
        print(f"Erro ao listar lotes no staging: {e}")
        return []

# Teste
print(f"Lotes JSON disponíveis: {detectar_lotes_disponiveis('json')}")
print(f"Lotes Parquet disponíveis: {detectar_lotes_disponiveis('parquet')}")

In [0]:
'''
Passo: criar função para publicar o próximo lote do staging para o inbox.
O que observar: move a pasta completa do lote, mantendo a estrutura de arquivos Spark.
Validar: o lote deve desaparecer do staging e aparecer no inbox após a execução.
Sinal de erro: copiar em vez de mover, duplicando arquivos e confundindo o Auto Loader.
'''

def publicar_proximo_lote(formato: str) -> dict:
    '''Publica o próximo lote disponível do staging para o inbox.
    
    Args:
        formato: 'json' ou 'parquet'
    
    Returns:
        Dicionário com status da operação e informações do lote.
    '''
    if formato not in ['json', 'parquet']:
        return {'sucesso': False, 'mensagem': "Formato deve ser 'json' ou 'parquet'"}
    
    # Detecta lotes disponíveis
    lotes_disponiveis = detectar_lotes_disponiveis(formato)
    
    if not lotes_disponiveis:
        ultimo = detectar_ultimo_lote_publicado(formato)
        return {
            'sucesso': False,
            'mensagem': f'✓ Dados já estão atualizados! Último lote publicado: {ultimo}',
            'ultimo_lote': ultimo
        }
    
    # Pega o próximo lote na sequência
    proximo_lote = lotes_disponiveis[0]
    lote_nome = f'lote_{proximo_lote:03d}'
    
    # Define paths origem e destino
    staging_path = JSON_STAGING_PATH if formato == 'json' else PARQUET_STAGING_PATH
    inbox_path = JSON_INBOX_PATH if formato == 'json' else PARQUET_INBOX_PATH
    
    origem = f'{staging_path}/{lote_nome}'
    destino = f'{inbox_path}/{lote_nome}'
    
    try:
        # Move o lote do staging para o inbox
        dbutils.fs.mv(origem, destino, recurse=True)
        
        return {
            'sucesso': True,
            'mensagem': f'✓ Lote {proximo_lote} ({formato.upper()}) publicado com sucesso!',
            'lote_publicado': proximo_lote,
            'lotes_restantes': len(lotes_disponiveis) - 1,
            'origem': origem,
            'destino': destino
        }
    
    except Exception as e:
        return {
            'sucesso': False,
            'mensagem': f'✗ Erro ao publicar lote {proximo_lote}: {str(e)}',
            'erro': str(e)
        }

# Teste
resultado = publicar_proximo_lote('json')
print(resultado['mensagem'])
if resultado['sucesso'] and 'lotes_restantes' in resultado:
    print(f"Lotes restantes no staging: {resultado['lotes_restantes']}")

In [0]:
'''
Passo: executar publicação com interface simples e feedback claro.
O que observar: cada execução publica apenas o próximo lote na sequência.
Validar: o bronze deve detectar automaticamente os novos arquivos no inbox.
Sinal de erro: tentar publicar lotes fora de ordem ou publicar todos de uma vez.
'''

# CONFIGURAÇÃO: escolha o formato a publicar
FORMATO = 'json'  # Altere para 'parquet' se necessário

print('='*70)
print(f'PUBLICAÇÃO PROGRESSIVA DE LOTES - FORMATO: {FORMATO.upper()}')
print('='*70)

# Detecta estado atual
ultimo = detectar_ultimo_lote_publicado(FORMATO)
disponiveis = detectar_lotes_disponiveis(FORMATO)

print(f'\nÚltimo lote publicado no inbox: {ultimo if ultimo > 0 else "nenhum"}')
print(f'Lotes disponíveis no staging: {disponiveis if disponiveis else "nenhum"}')

if disponiveis:
    print(f'\nPublicando próximo lote ({disponiveis[0]})...')
    resultado = publicar_proximo_lote(FORMATO)
    print(f'\n{resultado["mensagem"]}')
    
    if resultado['sucesso']:
        print(f'Origem:  {resultado["origem"]}')
        print(f'Destino: {resultado["destino"]}')
        print(f'\nLotes restantes no staging: {resultado["lotes_restantes"]}')
        
        if resultado['lotes_restantes'] > 0:
            print('\n💡 Execute esta célula novamente para publicar o próximo lote.')
        else:
            print('\n✓ Todos os lotes foram publicados!')
else:
    print(f'\n✓ Dados já estão atualizados!')

print('='*70)
print('INSTRUÇÕES:')
print('1. Execute esta célula para publicar o próximo lote')
print('2. Observe o bronze detectar automaticamente o novo arquivo')
print('3. Execute novamente para publicar mais lotes')
print(f"4. Para trocar de formato, altere FORMATO para 'json' ou 'parquet'")
print('='*70)